In [4]:
import data
import device_types as dt

data_conn = data.connect()
dt_conn = dt.connect()

In [ ]:


for f in dt.(dt_conn):
  print(dict(f))


{'id': 'A', 'name': 'Medical Center A', 'record_count': 6905}
{'id': 'B', 'name': 'Medical Center B', 'record_count': 639}
{'id': 'C', 'name': 'Medical Center C', 'record_count': 1021}
{'id': 'D', 'name': 'Medical Center D', 'record_count': 626}
{'id': 'E', 'name': 'Clinics and other sites', 'record_count': 1368}


# Challenge 1

In [20]:
conn = data.connect()

In [21]:
from collections import Counter

def norm(value: str | None) -> str:
    return " ".join((value or "").lower().split())

In [37]:
for f in conn.execute("SELECT id, record_count FROM facilities ORDER BY id").fetchall():
  records = conn.execute("SELECT * from records WHERE facility_id = ?", (f["id"])).fetchall()
  print(f"True record count {f["id"]}" if len(records) == f["record_count"] else False)

keys = Counter((norm(r["manufacturer"]), norm(r["model_name"])) for r in records)
print(f"True keys match total records" if sum(keys.values()) == len(records) else False)

True record count A
True record count B
True record count C
True record count D
True record count E
True keys match total records


In [ ]:
# sum of all blank records
sum(1 for r in records if not norm(r["manufacturer"]) or not norm(r["model_name"]))

0

In [25]:
default_pairs = {(r["manufacturer"], r["model_name"]) for r in records}

In [28]:
default_pairs

{('AARON MEDICAL INDUSTRIES', 'AARON 1250'),
 ('AARON MEDICAL INDUSTRIES', 'SMOKE SHARK'),
 ('ABLE', 'AP1310 BT'),
 ('ACCUCOLD', 'ARG6PV'),
 ('ACCUCOLD', 'ARS3PV'),
 ('ACCUCOLD', 'FF28LWH'),
 ('ACOUSTIC SYSTEMS', 'RE-125'),
 ('ADJUSTABLE FIXTURE', 'GOOSE-NECK'),
 ('AJUSTO EQUIP', 'GOOSENECK'),
 ('ALLIED HEALTHCARE', 'SCHUCO S5000'),
 ('AMBCO ELECTRONICS', '650A'),
 ('AMERICAN BIOTECH SUPPLY (ABS)', 'ABT-HC-19-TS'),
 ('AMERICAN BIOTECH SUPPLY (ABS)', 'ABT-HC-UCBI-0404'),
 ('AMERICAN BIOTECH SUPPLY (ABS)', 'ABT-HC-UCBI-0420A'),
 ('AMERICAN BIOTECH SUPPLY (ABS)', 'ABT-HC-UCFS-0440'),
 ('AMERICAN BIOTECH SUPPLY (ABS)', 'ABT-UCFS-0204G'),
 ('AMERICAN BIOTECH SUPPLY (ABS)', 'PH-ABT-12'),
 ('AMERICAN BIOTECH SUPPLY (ABS)', 'PH-ABT-19'),
 ('AMERICAN BIOTECH SUPPLY (ABS)', 'PH-ABT-23G'),
 ('AMERICAN BIOTECH SUPPLY (ABS)', 'PH-ABT-HC-10PG-RFC12'),
 ('AMERICAN BIOTECH SUPPLY (ABS)', 'PH-ABT-HC-23G'),
 ('AMERICAN BIOTECH SUPPLY (ABS)', 'PH-ABT-HC-S23G'),
 ('AMERICAN BIOTECH SUPPLY (ABS)', 'PH-ABT-

In [40]:
# check if default pairs equals norm'd pairs
len(default_pairs) == len(keys)

True

In [41]:
# singular entity pair
sum(1 for n in keys.values() if n == 1)

286

In [26]:
top = keys.most_common(2)

In [27]:
top

[(('welch allyn', 'green series 777'), 80),
 (('midmark', 'ritter 204-012'), 75)]

In [43]:
# checking for records with duplicate CEID
duped_ceid = conn.execute("SELECT ceid, COUNT(*) n FROM records GROUP BY ceid HAVING n > 1").fetchall()
dict(duped_ceid)

{'15789304': 2, '15789312': 2}

In [50]:

for d in duped_ceid:
  records = conn.execute(f"SELECT * FROM records WHERE CEID IS {d[0]}")
  for r in records:
    print(f"{dict(r)} \n")

{'record_id': 'C-1iscrs', 'facility_id': 'C', 'customer': 'Medical Center C', 'department': 'EMERGENCY', 'billing_unit': '16.145.68000', 'ceid': '15789304', 'serial_number': '18090772', 'manufacturer': 'CAREFUSION', 'model_name': 'ALARIS PUMP 8100', 'description_id': 'MODULES INFUSION PUMP', 'manufactured_date': '2018-09-01'} 

{'record_id': 'C-2ypd3u', 'facility_id': 'C', 'customer': 'Medical Center C', 'department': 'EMERGENCY', 'billing_unit': '16.145.68000', 'ceid': '15789304', 'serial_number': '18090585', 'manufacturer': 'CAREFUSION', 'model_name': 'ALARIS PUMP 8100', 'description_id': 'MODULES INFUSION PUMP', 'manufactured_date': '2026-03-11'} 

{'record_id': 'C-3ouimu', 'facility_id': 'C', 'customer': 'Medical Center C', 'department': 'MED SURG', 'billing_unit': '16.145.61140', 'ceid': '15789312', 'serial_number': '18091597', 'manufacturer': 'CAREFUSION', 'model_name': 'ALARIS PUMP 8100', 'description_id': 'MODULES INFUSION PUMP', 'manufactured_date': '2026-03-11'} 

{'record_id

In [57]:
# checking for entire duplicate records
entire_duped = conn.execute("SELECT COUNT(*) FROM (SELECT 1 FROM records GROUP BY facility_id, ceid, serial_number, manufacturer, model_name, department HAVING COUNT(*) > 1)").fetchone()[0]
# for r in entire_duped:
#   print(dict(entire_duped))
entire_duped

0

# Challenge 2

In [61]:
medc = ["A", "B", "C", "D"]


In [68]:
results = []

In [69]:
for cust in medc:
  results.append(conn.execute(f"SELECT customer, manufacturer, model_name, COUNT(*) as total_count FROM records WHERE customer = 'Medical Center {cust}' GROUP BY customer, manufacturer, model_name ORDER BY customer, total_count DESC"))

In [71]:
for f in results:
  for fac in f:
    print(dict(fac))

{'customer': 'Medical Center A', 'manufacturer': 'CAREFUSION', 'model_name': 'BD ALARIS PCU', 'total_count': 413}
{'customer': 'Medical Center A', 'manufacturer': 'PHILIPS MEDICAL SYSTEMS', 'model_name': 'INTELLIVUE X3', 'total_count': 178}
{'customer': 'Medical Center A', 'manufacturer': 'PHILIPS MEDICAL SYSTEMS', 'model_name': 'INTELLIVUE X3 DOCKING STATION', 'total_count': 176}
{'customer': 'Medical Center A', 'manufacturer': 'PHILIPS MEDICAL SYSTEMS', 'model_name': 'INTELLIVUE MX40', 'total_count': 167}
{'customer': 'Medical Center A', 'manufacturer': 'STRYKER MEDICAL', 'model_name': 'S3 MEDSURG', 'total_count': 139}
{'customer': 'Medical Center A', 'manufacturer': 'PHILIPS MEDICAL SYSTEMS', 'model_name': 'INTELLIVUE MULTI-MEASUREMENT MODULE MMX', 'total_count': 121}
{'customer': 'Medical Center A', 'manufacturer': 'WELCH ALLYN', 'model_name': 'SURETEMP PLUS 690', 'total_count': 111}
{'customer': 'Medical Center A', 'manufacturer': 'GENTEC', 'model_name': '881VR-300-OH-A', 'total_c

In [78]:
counts = {}
for fac in medc:
  rows = conn.execute("SELECT lower(trim(manufacturer)) as ma, lower(trim(model_name)) as mn, COUNT(*) AS n FROM records WHERE facility_id = ? GROUP BY manufacturer, model_name", (fac)).fetchall()
  counts[fac] = {(r["ma"], r["mn"]): r["n"] for r in rows}
print({fac: len(counts[fac]) for fac in medc})
print(counts)

{'A': 1326, 'B': 281, 'C': 446, 'D': 296}
{'A': {('3gen', 'dermlite dl100'): 1, ('3gen', 'dermlite dl200hr'): 3, ('3gen', 'dermlite dl4'): 1, ('3m healthcare', 'attest 490 auto-reader'): 2, ('3m healthcare', 'attest 490h autoreader'): 1, ('3m healthcare', 'attest mini auto-reader'): 2, ('3m healthcare', 'bair hugger 775'): 26, ('3m healthcare', 'ranger 245'): 4, ('3m healthcare', 'versaflo m-300'): 7, ('3m healthcare', 'versaflo tr-600'): 7, ('a-dec', 'pac-i (portable) model 3420'): 4, ('a-m systems', '3 liter syringe'): 5, ('abbott laboratories', 'i-stat printer'): 1, ('abbott medical optics', 'ionic rf generator'): 2, ('abiomed', 'impella controller'): 3, ('able', 'ap1310 bt'): 3, ('accriva diagnostics', 'avoximeter 1000e'): 1, ('accucold', 'ff28lwh'): 1, ('accucold', 'fs24lmed2'): 1, ('accuvein', 'av400'): 2, ('acist medical systems', 'cvi control panel'): 1, ('acist medical systems', 'cvi injector head'): 5, ('acist medical systems', 'rxi system console'): 2, ('acoustic systems', '

In [90]:
counts = {}
for fac in medc:
  rows = conn.execute("SELECT manufacturer, model_name, COUNT(*) AS n FROM records WHERE facility_id = ? GROUP BY manufacturer, model_name", (fac)).fetchall()
  counts[fac] = {(r["manufacturer"], r["model_name"]): r["n"] for r in rows}
print({fac: len(counts[fac]) for fac in medc})
print(counts)


{'A': 1326, 'B': 281, 'C': 446, 'D': 296}
{'A': {('3GEN', 'DERMLITE DL100'): 1, ('3GEN', 'DERMLITE DL200HR'): 3, ('3GEN', 'DERMLITE DL4'): 1, ('3M HEALTHCARE', 'ATTEST 490 AUTO-READER'): 2, ('3M HEALTHCARE', 'ATTEST 490H AUTOREADER'): 1, ('3M HEALTHCARE', 'ATTEST MINI AUTO-READER'): 2, ('3M HEALTHCARE', 'BAIR HUGGER 775'): 26, ('3M HEALTHCARE', 'RANGER 245'): 4, ('3M HEALTHCARE', 'VERSAFLO M-300'): 7, ('3M HEALTHCARE', 'VERSAFLO TR-600'): 7, ('A-DEC', 'PAC-I (PORTABLE) MODEL 3420'): 4, ('A-M SYSTEMS', '3 LITER SYRINGE'): 5, ('ABBOTT LABORATORIES', 'I-STAT PRINTER'): 1, ('ABBOTT MEDICAL OPTICS', 'IONIC RF GENERATOR'): 2, ('ABIOMED', 'IMPELLA CONTROLLER'): 3, ('ABLE', 'AP1310 BT'): 3, ('ACCRIVA DIAGNOSTICS', 'AVOXIMETER 1000E'): 1, ('ACCUCOLD', 'FF28LWH'): 1, ('ACCUCOLD', 'FS24LMED2'): 1, ('ACCUVEIN', 'AV400'): 2, ('ACIST MEDICAL SYSTEMS', 'CVI CONTROL PANEL'): 1, ('ACIST MEDICAL SYSTEMS', 'CVI INJECTOR HEAD'): 5, ('ACIST MEDICAL SYSTEMS', 'RXI SYSTEM CONSOLE'): 2, ('ACOUSTIC SYSTEMS', '

In [91]:
shared = set(counts["A"])
for fac in ["B", "C", "D"]:
  shared = shared & set(counts[fac])
print(len(shared))

40


In [93]:
models = []

for manufacturer, model in shared:
  per_facility = {}
  for fac in medc:
    per_facility[fac] = counts[fac][(manufacturer, model)]
  models.append({
    "manufacturer": manufacturer,
    "model": model,
    "total": sum(per_facility.values()),
    "per_facility": per_facility
  })

In [94]:
models

[{'manufacturer': 'OLYMPUS AMERICA',
  'model': 'MAINTENANCE UNIT',
  'total': 6,
  'per_facility': {'A': 3, 'B': 1, 'C': 1, 'D': 1}},
 {'manufacturer': 'PHILIPS MEDICAL SYSTEMS',
  'model': 'EPIQ CVX',
  'total': 6,
  'per_facility': {'A': 3, 'B': 1, 'C': 1, 'D': 1}},
 {'manufacturer': 'OLYMPUS AMERICA',
  'model': 'EVIS EXERA III CV-190',
  'total': 17,
  'per_facility': {'A': 8, 'B': 1, 'C': 5, 'D': 3}},
 {'manufacturer': 'SMITHS MEDICAL',
  'model': 'MEDFUSION 3500',
  'total': 42,
  'per_facility': {'A': 36, 'B': 3, 'C': 1, 'D': 2}},
 {'manufacturer': 'PELSTAR',
  'model': 'HEALTH O METER 1100KL',
  'total': 11,
  'per_facility': {'A': 8, 'B': 1, 'C': 1, 'D': 1}},
 {'manufacturer': 'BIO-MEDICAL DEVICES',
  'model': 'MAXAIR SYSTEM',
  'total': 71,
  'per_facility': {'A': 59, 'B': 6, 'C': 1, 'D': 5}},
 {'manufacturer': 'STRYKER MEDICAL',
  'model': 'GYNNIE',
  'total': 5,
  'per_facility': {'A': 2, 'B': 1, 'C': 1, 'D': 1}},
 {'manufacturer': 'ORTHO CLINICAL DIAGNOSTICS',
  'model': 